In [1]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from os.path import exists

sys.path.append('../..')

In [3]:
import numpy as np
from loguru import logger
import torch

from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.monitor import Monitor

In [4]:
from vimms.Common import POSITIVE, load_obj, save_obj
from vimms.ChemicalSamplers import MZMLFormulaSampler, MZMLRTandIntensitySampler, \
    MZMLChromatogramSampler
from vimms.Roi import RoiBuilderParams

from vimms_gym.env import DDAEnv
from vimms_gym.chemicals import generate_chemicals
from vimms_gym.evaluation import run_method

# 1. Parameters

In [5]:
n_chemicals = (2000, 5000)
mz_range = (70, 1000)
rt_range = (0, 1440)
intensity_range = (1E4, 1E20)

In [6]:
min_mz = mz_range[0]
max_mz = mz_range[1]
min_rt = rt_range[0]
max_rt = rt_range[1]
min_log_intensity = np.log(intensity_range[0])
max_log_intensity = np.log(intensity_range[1])

In [7]:
isolation_window = 0.7
N = 10
rt_tol = 120
mz_tol = 10
min_ms1_intensity = 5000
ionisation_mode = POSITIVE

enable_spike_noise = True
noise_density = 0.1
noise_max_val = 1E3

In [8]:
mzml_filename = '../fullscan_QCB.mzML'
samplers = None
samplers_pickle = 'samplers_fullscan_QCB.mzML.p'
if exists(samplers_pickle):
    logger.info('Loaded %s' % samplers_pickle)
    samplers = load_obj(samplers_pickle)
    mz_sampler = samplers['mz']
    ri_sampler = samplers['rt_intensity']
    cr_sampler = samplers['chromatogram']
else:
    logger.info('Creating samplers from %s' % mzml_filename)
    mz_sampler = MZMLFormulaSampler(mzml_filename, min_mz=min_mz, max_mz=max_mz)
    ri_sampler = MZMLRTandIntensitySampler(mzml_filename, min_rt=min_rt, max_rt=max_rt,
                                           min_log_intensity=min_log_intensity,
                                           max_log_intensity=max_log_intensity)
    roi_params = RoiBuilderParams(min_roi_length=3, at_least_one_point_above=1000)
    cr_sampler = MZMLChromatogramSampler(mzml_filename, roi_params=roi_params)
    samplers = {
        'mz': mz_sampler,
        'rt_intensity': ri_sampler,
        'chromatogram': cr_sampler
    }
    save_obj(samplers, samplers_pickle)

2022-05-17 08:49:13.374 | INFO     | __main__:<cell line: 4>:5 - Loaded samplers_fullscan_QCB.mzML.p


In [9]:
params = {
    'chemical_creator': {
        'mz_range': mz_range,
        'rt_range': rt_range,
        'intensity_range': intensity_range,
        'n_chemicals': n_chemicals,
        'mz_sampler': mz_sampler,
        'ri_sampler': ri_sampler,
        'cr_sampler': cr_sampler,
    },
    'noise': {
        'enable_spike_noise': enable_spike_noise,
        'noise_density': noise_density,
        'noise_max_val': noise_max_val,
        'mz_range': mz_range
    },
    'env': {
        'ionisation_mode': ionisation_mode,
        'rt_range': rt_range,
        'isolation_window': isolation_window,
        'mz_tol': mz_tol,
        'rt_tol': rt_tol,
    }
}

In [10]:
max_peaks = 200
in_dir = 'results'

In [11]:
deterministic = True
cpu_limit = 40

# 2. Train PPO

In [12]:
def make_env(rank, seed=0):
    def _init():
        env = DDAEnv(max_peaks, params)
        env.seed(rank)
        env = Monitor(env)        
        return env
    set_random_seed(seed)
    return _init

### Using DDAEnv

In [13]:
env = DDAEnv(max_peaks, params)
check_env(env)

In [14]:
env_name = 'DDAEnv'

### Parameter set 1

In [15]:
# default parameters
# learning_rate = 0.0003
# batch_size = 64
# n_steps = 2048
# ent_coef = 0.0
# gamma = 0.99
# gae_lambda = 0.95

In [16]:
# parameter set 1
learning_rate = 0.0003
batch_size = 512
n_steps = 2048
ent_coef = 0.001
gamma = 0.90
gae_lambda = 0.90
hidden_nodes = 512
total_timesteps = 100E6

net_arch = [dict(pi=[hidden_nodes, hidden_nodes], vf=[hidden_nodes, hidden_nodes])]
policy_kwargs = dict(net_arch=net_arch)

In [17]:
model_name = 'PPO'
fname = '%s/%s_%s.zip' % (in_dir, env_name, model_name)

In [18]:
fname

'results/DDAEnv_PPO.zip'

In [19]:
num_cpu = int(cpu_limit / 2)
torch.set_num_threads(num_cpu)
env = SubprocVecEnv([make_env(i) for i in range(num_cpu)])

tensorboard_log = './%s/%s_%s_tensorboard' % (in_dir, env_name, model_name)

model = PPO('MultiInputPolicy', env, 
            tensorboard_log=tensorboard_log, 
            learning_rate=learning_rate, batch_size=batch_size, n_steps=n_steps, 
            ent_coef=ent_coef, gamma=gamma, gae_lambda=gae_lambda, policy_kwargs=policy_kwargs)
model.learn(total_timesteps=total_timesteps)
model.save(fname)

# 3. Train DQN

In [20]:
# model_name = 'DQN'
# fname = '%s/%s_%s.zip' % (in_dir, env_name, model_name)

In [21]:
# # original parameters
# learning_rate = 0.0001
# batch_size = 32
# gamma = 0.99
# exploration_fraction = 0.1
# exploration_initial_eps = 1.0
# exploration_final_eps = 0.05
# hidden_nodes = 64
# total_timesteps = 5E6

# net_arch = [hidden_nodes, hidden_nodes]
# policy_kwargs = dict(net_arch=net_arch)

In [22]:
# # modified parameters
# learning_rate = 0.0001
# batch_size = 512
# gamma = 0.90
# exploration_fraction = 0.25
# exploration_initial_eps = 1.0
# exploration_final_eps = 0.10
# hidden_nodes = 512
# total_timesteps = 5E6

# net_arch = [hidden_nodes, hidden_nodes]
# policy_kwargs = dict(net_arch=net_arch)

In [23]:
# num_cpu = cpu_limit
# torch.set_num_threads(num_cpu)
# env = DDAEnv(max_peaks, params)
# env = Monitor(env)
# env = DummyVecEnv([lambda: env])

# tensorboard_log = './%s/%s_%s_tensorboard' % (in_dir, env_name, model_name)

# model = DQN('MultiInputPolicy', env, 
#             tensorboard_log=tensorboard_log, 
#             learning_rate=learning_rate, batch_size=batch_size, gamma=gamma,
#             exploration_fraction=exploration_fraction, exploration_initial_eps=exploration_initial_eps, exploration_final_eps=exploration_final_eps, 
#             policy_kwargs=policy_kwargs)
# model.learn(total_timesteps=total_timesteps)
# model.save(fname)